In [ ]:
# Policy Validation Notebook

This notebook validates and inspects policies data using the SDK.

**Important**: Analysts should ONLY use SDK methods. Do NOT access database directly.

## Setup

First, install the package:
```bash
pip install -e .
```


In [ ]:
# Install package first: pip install -e .
import pandas as pd
from src.sdk import PoliciesAnalyst

# Use SDK to access data - do NOT access database directly
analyst = PoliciesAnalyst(db_path="../warehouse.db")

print("PoliciesAnalyst SDK initialized")
print("Use SDK methods to access data from database")


## Load Policies Data

Load policies using SDK.


In [ ]:
# Load policies from database using SDK
policies = analyst.load_from_database(layer="silver")

# Convert to DataFrame for analysis
df = analyst.to_dataframe(policies)

print(f"Loaded {len(policies)} policies from database")
print(f"\nDataFrame shape: {df.shape}")
df.head()


## Get Summary Statistics

Use SDK method for statistics.


In [ ]:
# Get summary statistics - SDK does all calculations
stats = analyst.get_summary_statistics(policies)

print("Summary Statistics:")
print(f"  Total Policies: {stats['total_policies']}")
print(f"  Total Coverage: ${stats['total_coverage']:,.2f}")
print(f"  Average Coverage: ${stats['average_coverage']:,.2f}")
print(f"\nBy Status:")
for status, count in stats['by_status'].items():
    print(f"  {status}: {count}")


## Data Quality Checks

Simple pandas operations (no business logic).


In [ ]:
# Simple data quality checks using pandas
print("Null values:")
print(df.isnull().sum())

print("\nStop loss limit statistics:")
print(df['stop_loss_limit'].describe())

print("\nStatus distribution:")
print(df['status'].value_counts())

# Use SDK to get active policies (business logic in SDK)
active = analyst.get_active_policies(policies)
print(f"\nActive policies: {len(active)}")


## Filter Active Policies

Use SDK method - business logic is encapsulated.


In [ ]:
# Get active policies using SDK
active = analyst.get_active_policies(policies)
print(f"Active policies: {len(active)}")

if active:
    active_coverage = analyst.get_total_coverage(active)
    print(f"Total active coverage: ${active_coverage:,.2f}")
    
    # Convert to DataFrame for display
    active_df = analyst.to_dataframe(active)
    active_df[['policy_id', 'employer_id', 'stop_loss_limit', 'status']]


## Policies by Employer

Use SDK method for lookup.


In [ ]:
# Simple pandas groupby for visualization (no business logic)
policies_by_employer = df.groupby('employer_id')['stop_loss_limit'].agg(['sum', 'mean', 'count']).round(2)
policies_by_employer.columns = ['total_coverage', 'avg_coverage', 'policy_count']
policies_by_employer.sort_values('total_coverage', ascending=False)


## High Coverage Policies

Simple pandas filtering for display (business logic in SDK).


In [ ]:
# Simple pandas filtering for visualization
high_coverage = df[df['stop_loss_limit'] > 2000000].sort_values('stop_loss_limit', ascending=False)
print(f"High coverage policies (>$2M): {len(high_coverage)}")
high_coverage[['policy_id', 'employer_id', 'stop_loss_limit', 'aggregate_deductible', 'specific_deductible', 'status']]


## Notes

- All data access is through SDK methods
- No direct database access
- Business logic is in SDK, not notebooks
- Use pandas only for simple visualization


In [ ]:
print("Analysis complete")
print("\nRemember: Always use SDK methods, never access database directly!")
